<a href="https://colab.research.google.com/github/100522094/aprendizaje-p1-100522234-100522094/blob/main/parte2_100522094_100522234.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRIMERA PRÁCTICA - ENTRENAMENTO Y ANÁLISIS DEL MODELO**

### **PREDICCIÓN DE SUBSCRIPCIÓN A UN PRODUCTO BANCARIO - Cuaderno 2**

### **Aprendizaje Automático - Grado en Ingeniería Informática - Curso 2025/26**

### **Autores**

<img src="https://www.elearningmedia.es/sites/default/files/inline-images/logo_UC3M.png" align="right" width="200" style="margin-left: 20px;">

Grupo 82 - Equipo 05

- Carmen Peláez Martín - 100522094
- Ana Sanz del Collado - 100522234


Link repositorio Git Hub:

https://github.com/100522094/aprendizaje-p1-100522234-100522094


Antes de comenzar, forzamos la versión de scikit-learn para asegurar que es compatible con el modelo guardado .pkl


**IMPORTANTE**: Asegurate de que los achivos "bank_competition.pkl" y "modelo_final.pkl" están cargados correctamente

In [ ]:
!pip install scikit-learn==1.8.0

### **Objetivo de este cuaderno**
El propósito de este segundo cuaderno es totalmete operativo. Aquí no vamos a entrenar ningún algoritmo ni a explorar los datos. Vamos a simular que el banco nos ha entregado una lista de clientes nuevos (el archivo de competición) y no sabemos si van a suscribir el depósito o no

Los pasos que vamos a seguir son:
1. Cargar los datos nuevos de los clientes
2. Aplicarles la misma transformación manual que hicimos en el primer cuaderno (la variable `pdays`)
3. Cargar el "cerebro" de nuestro mejor modelo (el archivo `modelo_final.pkl` exportado en el cuaderno 1)
4. Generar las predicciones y guardarlas en un archivo `predicciones.csv`

In [ ]:
# importamos las dos únicas herramientas que necesitamos aqui
import pandas as pd
import joblib

# 1. CARGAMOS LOS DATOS DE LA COMPETICIÓN
df_competicion = pd.read_pickle('bank_competition.pkl')

print(f"Datos de competición cargados correctamente. Tenemos {len(df_competicion)} clientes nuevos para analizar")

Datos de competición cargados correctamente. Tenemos 162 clientes nuevos para analizar


### **Preparación de los datos (Igualar la estructura)**

Nuestro modelo guardado (el Pipeline) sabe escalar números y transformar categorías él solo. Sin embargo, en el Cuaderno 1, nosotros creamos manualmente una variable nueva llamada `was_previously_contacted` a partir de `pdays` y borramos la original

Si le pasamos los datos al modelo sin hacer esto primero, el modelo no reconocerá las columnas. Por tanto, tenemos que hacer exactamente lo mismo a estos clientes nuevos

In [ ]:
# 2. TRANSFORMACIÓN MANUAL DE VARIABLES

# como hemos explicado, creamos una variable binaria para 'pdays' porque
# el valor -1 no tiene magnitud numerica real, sino que representa otra
# categoría:

# 0 = nunca contactado (-1)
# 1 = contactado previamente (>0)

# creamos variable binaria
df_competicion['was_previously_contacted'] = (df_competicion['pdays'] != -1).astype(int)

# limpiamos los -1 poniendolos a 0 (como hicimos en el Cuaderno 1)
df_competicion['pdays_clean'] = df_competicion['pdays'].replace(-1, 0)

# Borramos la variable original 'pdays' porque el modelo ya no la espera
X_nuevos_clientes = df_competicion.drop(columns=['pdays'])

print("Datos preparados. Las columnas coinciden exactamente con las que aprendió el modelo")

Datos preparados. Las columnas coinciden exactamente con las que aprendió el modelo


### **Predicción y Exportación de Resultados**

Ahora, cogemos el modelo guardado, le pasamos la tabla de los nuevos clientes y le pedimos que nos diga si dirán "yes" o "no"

Finalmente, guardamos las respuestas en el archivo "predicciones.csv" como se pide en el enunciado

In [ ]:
# 3. CARGAR EL MODELO
modelo_cargado = joblib.load('modelo_final.pkl')
print("Modelo SVM cargado exitosamente")

# 4. HACER LAS PREDICCIONES
# le pasamos los datos al pipeline y se encarga de rellenar nulos, escalar y predecir
predicciones_finales = modelo_cargado.predict(X_nuevos_clientes)

# 5. GUARDAR LOS RESULTADOS EN UN CSV
# convertimos la lista de "yes" y "no" en una tabla de una sola columna llamada 'deposit'
df_resultados = pd.DataFrame(predicciones_finales, columns=['deposit'])

# exportamos a CSV. El parámetro index=False es muy importante para no meter una columna extra de números
df_resultados.to_csv('predicciones.csv', index=False)

print("Predicciones generadas. El archivo 'predicciones.csv' ya está listo\n")

Modelo SVM cargado exitosamente
Predicciones generadas. El archivo 'predicciones.csv' ya está listo



## **Conclusiones generales - Cuaderno 2**


Con la generación del archivo `predicciones.csv`, damos por concluida la fase de predicción sobre datos nuevos. Este segundo cuaderno nos deja las siguientes conclusiones:

1. **Separación entre Entrenamiento y Producción:** Hemos demostrado que no es necesario reentrenar un modelo cada vez que llegan datos nuevos. Al guardar nuestro mejor modelo (la SVM) en el primer cuaderno usando *joblib*, hemos podido "despertar al modelo" aquí de forma instantánea. Esto simula cómo trabajan los bancos en el mundo real: el modelo se entrena una vez y se usa miles de veces

2. **Consistencia en el preprocesamiento:** El modelo guardado (nuestro Pipeline) sabe escalar y codificar variables por sí solo, pero no sabe hacer transformaciones de lógica de negocio que nosotros inventamos. Por eso, ha sido crucial aplicar a los clientes nuevos la misma transformación manual sobre la variable `pdays` (creando `was_previously_contacted`) que hicimos en el entrenamiento. Si la estructura de los datos nuevos no es idéntica a la que el modelo aprendió, la predicción falla

3. **Control de versiones (Aviso InconsistentVersionWarning):** Durante el desarrollo nos encontramos advertencias de compatibilidad. Esto nos ha enseñado que el entorno donde predecimos debe ser exactamente el mismo que el entorno donde entrenamos. Al forzar a Colab a usar *scikit-learn==1.8.0* en este cuaderno, aseguramos que el archivo .pkl se lea sin corrupciones ni errores

4. **Predicción a ciegas (Simulación real):** A diferencia de la Fase 6 del primer cuaderno (donde evaluábamos contra un conjunto de Test y veíamos la nota), aquí hemos predecido a ciegas. El dataset bank_competition no contiene la variable objetivo `deposit`, por lo que no podemos medir nuestro Accuracy real. Confiamos plenamente en que la generalización del 84.84% obtenida en validación cruzada se mantenga para estos nuevos clientes